# Robustez y comparación estadística

Con 140 modelos hay $\binom{140}{2} = 9\,730$ pares posibles: probarlos todos con $\alpha = 0{,}05$ daría ≈ 487 falsos positivos por azar. La comparación es **jerárquica**: primero una prueba global y solo después comparaciones dirigidas con corrección por comparaciones múltiples y tamaño del efecto.

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path[:0] = [os.path.abspath(".."), os.path.abspath("../.."), os.path.abspath("../../entregable1")]  # e3, src y e1
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import contextlib, io  # Silenciar un diagnóstico de joblib en Windows
from joblib.externals.loky import cpu_count  # Conteo de núcleos
with contextlib.redirect_stderr(io.StringIO()):  # joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché)
import pickle  # Lectura de resultados
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Paneles
from IPython.display import Markdown, display  # Texto enriquecido
from e1.figuras import mostrar, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO  # Estilo de figuras (tema claro del informe)
from e3 import config as c3, datos as dd, modelos as mo, optimizadores as op, experimento as ex  # Código del entregable
c3.fijar_semilla()  # Semilla global (42)
pd.set_option("display.width", 170)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
COLORES = {"Grid Search": GRIS, "Random Search": AMBAR, "Bayesiana (Optuna TPE)": AZUL, "Genética (DEAP)": VERDE, "Successive Halving": MORADO}  # Color por optimizador
import joblib  # Lectura
from sklearn.metrics import average_precision_score, roc_auc_score  # Métricas
from e3 import estadistica as st, evaluacion as ev  # Pruebas y métricas
tabla = pd.read_csv(c3.RESULTADOS / "tabla_maestra.csv")  # Tabla maestra
tr, va, te = dd.cargar()  # Periodos
y_te = te["y"].to_numpy()  # Objetivo de la prueba
semanas = te["time_idx"].to_numpy()  # Semana de cada fila (bloques)
finales = {}  # Predicciones finales
for f in sorted(ex.FINALES.glob("*.pkl")):  # Reajustes finales
    r = pickle.load(open(f, "rb"))  # Predicciones
    finales[(r["tarea"], r["modelo"], r["balanceo"], r["optimizador"])] = r  # Indexado
print(f"Reajustes finales: {len(finales)}")  # 140

Reajustes finales: 140


## 1. Robustez: media ± desviación en el bucle externo y sensibilidad a la semilla

In [2]:
clf = tabla[tabla["tarea"] == "clf"]  # Clasificación
ext = clf.groupby(["modelo", "balanceo", "optimizador"])["AUC-PR"].agg(["mean", "std"])  # Media y desviación de los 3 pliegues
mej = ext.reset_index().sort_values("mean", ascending=False).groupby("modelo").first()  # Mejor combinación de cada modelo
mej["media ± desv."] = mej.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)  # Formato
mej.sort_values("mean", ascending=False)[["balanceo", "optimizador", "media ± desv."]]

,balanceo,optimizador,media ± desv.
modelo,,,
XGBoost,Sin balanceo,Genética (DEAP),0.679 ± 0.104
Random Forest,class_weight balanced,Genética (DEAP),0.678 ± 0.107
SVM,Sin balanceo,Genética (DEAP),0.671 ± 0.107
Regresión logística,Sin balanceo,Genética (DEAP),0.653 ± 0.112
KNN,Sin balanceo,Random Search,0.651 ± 0.116
Árbol de decisión,class_weight balanced,Bayesiana (Optuna TPE),0.634 ± 0.115
Naive Bayes,ADASYN,Bayesiana (Optuna TPE),0.546 ± 0.137


In [3]:
rob = joblib.load(c3.RESULTADOS / "robustez_semillas.joblib")  # 5 semillas por modelo
sem = rob.groupby("modelo").agg(**{"AUC-PR media (5 semillas)": ("AUC-PR", "mean"), "desv. por semilla": ("AUC-PR", "std"),
                                   "rango": ("AUC-PR", lambda s: s.max() - s.min()), "AUC-ROC desv. por semilla": ("AUC-ROC", "std")}).sort_values("AUC-PR media (5 semillas)", ascending=False)
sem["desv. entre pliegues externos"] = mej["std"]  # Variabilidad por partición
sem

,AUC-PR media (5 semillas),desv. por semilla,rango,AUC-ROC desv. por semilla,desv. entre pliegues externos
modelo,,,,,
XGBoost,0.8214,0.0015,0.0034,0.0004,0.1041
SVM,0.8197,0.0012,0.0031,0.0006,0.1068
Random Forest,0.8187,0.0007,0.0019,0.0002,0.1068
Regresión logística,0.8061,0.0006,0.0016,0.0003,0.1120
KNN,0.7995,0.0011,0.0028,0.0004,0.1160
Árbol de decisión,0.7851,0.0078,0.0212,0.0010,0.1146
Naive Bayes,0.7186,0.0007,0.0019,0.0008,0.1374


**Interpretación.** La variabilidad por **semilla** (submuestra, remuestreo e inicialización) es pequeña: una desviación de 0,0006–0,0015 de AUC-PR en la prueba para todos salvo el árbol único (0,008, el más inestable, como se espera de un solo árbol). La variabilidad **entre pliegues externos** es unas 100 veces mayor (≈ 0,10), porque cada año tiene otra prevalencia. Las diferencias entre los tres mejores modelos (XGBoost, SVM y Random Forest: 0,821, 0,820 y 0,819) son del mismo orden que la variabilidad por semilla: **no son mayores que la variabilidad intrínseca**, lo que exige pruebas formales antes de declarar un ganador.

## 2. Clasificación — Etapa 1: prueba global de Friedman

Bloques = las 100 semanas de la prueba (cada semana es un problema de ordenar el riesgo de 376 municipios); en cada semana se ordenan los modelos por AUC-ROC. Se aplica a los **7 modelos** (mejor combinación de cada uno) y a las **112 combinaciones**.

In [4]:
def auc_semanal(claves):  # Matriz semanas × modelos con el AUC-ROC de cada semana
    cols = {}  # Columnas
    for k in claves:  # Modelos
        s = finales[k]["p_te"].astype(float)  # Puntaje
        cols[k] = {w: roc_auc_score(y_te[semanas == w], s[semanas == w]) for w in np.unique(semanas) if 0 < y_te[semanas == w].mean() < 1}  # Semanas con ambas clases
    return pd.DataFrame(cols)  # Tabla

siete = [("clf", m, r["balanceo"], r["optimizador"]) for m, r in mej.iterrows()]  # Mejor combinación de cada modelo
A7 = auc_semanal(siete)  # 100 semanas × 7 modelos
A7.columns = [k[1] for k in siete]  # Nombres
f7 = st.friedman(A7)  # Friedman
pd.DataFrame({"Estadístico de Friedman χ²_F": [f7["chi2"]], "g.l.": [f7["gl"]], "valor p": [f7["p"]], "F de Iman-Davenport": [f7["F"]], "p (F)": [f7["p_F"]],
              "bloques (semanas)": [f7["N"]], "Conclusión": ["Se rechaza H0: hay diferencias" if f7["p"] < 0.05 else "No se rechaza H0"]})

,Estadístico de Friedman χ²_F,g.l.,valor p,F de Iman-Davenport,p (F),bloques (semanas),Conclusión
0,437.5811,6,0.0000,266.7209,0.0000,100,Se rechaza H0: hay diferencias


## 3. Clasificación — Etapa 2: Nemenyi y diagrama de Diferencia Crítica

In [5]:
def diagrama_cd(rangos, cd, titulo, alto=None):  # Diagrama de diferencia crítica (una fila por modelo)
    """Cada modelo es un punto en su rango medio con una barra de longitud CD centrada en él (± CD/2): si las barras de dos
    modelos no se tocan, su diferencia de rango supera la CD y son significativamente distintos según Nemenyi. Los modelos
    cuyas barras tocan la franja del mejor (rojo) son estadísticamente indistinguibles de él."""
    r = rangos.sort_values()  # Rangos (mejor arriba)
    nombres = list(r.index)  # Etiquetas
    indist = r - r.iloc[0] < cd  # Indistinguibles del mejor
    fig = go.Figure()  # Figura
    fig.add_vrect(x0=r.iloc[0], x1=r.iloc[0] + cd, fillcolor=ROJO, opacity=0.10, line_width=0)  # Franja de la CD desde el mejor
    fig.add_scatter(x=r.values, y=nombres, mode="markers", marker=dict(size=10, color=np.where(indist, ROJO, AZUL)),  # Rango medio
                    error_x=dict(type="constant", value=cd / 2, color="#94a3b8", thickness=2), showlegend=False,
                    hovertemplate="%{y}: rango %{x:.2f}<extra></extra>")
    fig.update_layout(title=titulo, xaxis_title=f"rango medio (1 = mejor) · barras de longitud CD = {cd:.2f}", yaxis=dict(autorange="reversed"),
                      margin=dict(l=320))  # Formato
    mostrar(fig, alto or max(320, 34 * len(r) + 120))  # Muestra

cd7 = st.diferencia_critica(f7["k"], f7["N"])  # Diferencia crítica
diagrama_cd(f7["rangos"], cd7, f"Diagrama de Diferencia Crítica — 7 modelos (Nemenyi, α = 0,05, N = {f7['N']} semanas)")  # Diagrama
f7["rangos"].to_frame("rango medio")

,rango medio
XGBoost,1.8400
SVM,2.1000
Random Forest,2.5500
Regresión logística,4.6850
KNN,4.9500
Árbol de decisión,5.1750
Naive Bayes,6.7000


In [6]:
todas = [k for k in finales if k[0] == "clf"]  # 112 combinaciones
A112 = auc_semanal(todas)  # 100 semanas × 112
A112.columns = [f"{k[1]} · {k[2].replace('class_weight balanced', 'pesos')} · {k[3].split(' ')[0]}" for k in todas]  # Nombres cortos
f112 = st.friedman(A112)  # Friedman sobre todo el universo
cd112 = st.diferencia_critica(f112["k"], f112["N"])  # CD con k = 112
print(f"112 combinaciones: χ²_F = {f112['chi2']:.1f}, g.l. = {f112['gl']}, p = {f112['p']:.2e} · CD de Nemenyi = {cd112:.2f}")
top = f112["rangos"].head(20)  # Las 20 mejores
lider = top.iloc[0]  # Mejor rango
indist = f112["rangos"][f112["rangos"] - lider < cd112]  # Indistinguibles del líder
diagrama_cd(top, cd112, f"Diferencia Crítica — 20 mejores de 112 combinaciones (CD = {cd112:.2f})")  # Diagrama
print(f"Combinaciones estadísticamente indistinguibles de la mejor ({top.index[0]}): {len(indist)} de 112")

112 combinaciones: χ²_F = 7673.0, g.l. = 111, p = 0.00e+00 · CD de Nemenyi = 19.98


Combinaciones estadísticamente indistinguibles de la mejor (XGBoost · pesos · Random): 30 de 112


**Interpretación.** Friedman rechaza con claridad que los 7 modelos sean equivalentes ($\chi^2_F = 437{,}6$, $p \approx 0$). Con Nemenyi ($CD = 0{,}90$ para $k = 7$ y $N = 100$), **XGBoost (1,84), SVM (2,10) y Random Forest (2,55) forman un grupo estadísticamente indistinguible**; regresión logística, KNN y el árbol forman un segundo grupo (4,7–5,2), y Naive Bayes queda solo al final (6,7). Sobre las 112 combinaciones, la CD crece a ≈ 20 (con $k = 112$ es muy exigente) y **30 combinaciones** son indistinguibles de la mejor: casi todas de XGBoost, Random Forest y SVM con distintos balanceos y optimizadores. El diagrama resume en una figura lo que serían 9 730 comparaciones.

## 4. Comparaciones dirigidas: DeLong con corrección de Holm, bootstrap BCa y delta de Cliff

Se comparan los **tres mejores modelos** de la Etapa 1 y la regresión logística como referencia lineal.

In [7]:
orden = f7["rangos"].index.tolist()  # Modelos por rango
candidatos = orden[:3] + (["Regresión logística"] if "Regresión logística" not in orden[:3] else [])  # Subconjunto reducido
punt = {m: finales[k]["p_te"].astype(float) for k in siete for m in [k[1]] if m in candidatos}  # Puntajes
pares = [(a, b) for i, a in enumerate(candidatos) for b in candidatos[i + 1:]]  # Pares
dl = st.delong_holm(y_te, punt, pares)  # DeLong + Holm
mun = te["COD_MUN_N"].to_numpy()  # Conglomerados
ic, cliff = [], []  # Bootstrap BCa y Cliff
for a, b in pares:  # Cada par
    obs, lo, hi = st.bca_municipios(np.c_[y_te, punt[a], punt[b]], np.zeros(len(y_te)), mun,
                                     lambda M, _: roc_auc_score(M[:, 0], M[:, 1]) - roc_auc_score(M[:, 0], M[:, 2]), B=1000)  # ΔAUC con BCa por municipio
    ic.append((lo, hi))  # IC
    cliff.append(st.delta_cliff(A7[a], A7[b]))  # Dominancia semanal
dl["IC 95 % BCa (municipios)"] = [f"[{lo:.4f}; {hi:.4f}]" for lo, hi in ic]  # IC bootstrap
dl["δ de Cliff (AUC semanal)"] = cliff  # Tamaño del efecto
dl["magnitud"] = [st.magnitud_cliff(x) for x in cliff]  # Etiqueta
dl

,modelo A,modelo B,ΔAUC (A − B),"IC 2,5 %","IC 97,5 %",p crudo,p Holm,IC 95 % BCa (municipios),δ de Cliff (AUC semanal),magnitud
0,XGBoost,SVM,0.0010,0.0000,0.0019,0.0475,0.0949,[-0.0002; 0.0023],0.0134,despreciable
1,XGBoost,Random Forest,0.0014,0.0009,0.0018,0.0000,0.0000,[0.0007; 0.0020],0.0446,despreciable
2,XGBoost,Regresión logística,0.0121,0.0107,0.0135,0.0000,0.0000,[0.0101; 0.0145],0.3384,mediano
3,SVM,Random Forest,0.0004,-0.0006,0.0014,0.4257,0.4257,[-0.0010; 0.0015],0.0252,despreciable
4,SVM,Regresión logística,0.0111,0.0099,0.0124,0.0000,0.0000,[0.0091; 0.0134],0.3242,pequeño
5,Random Forest,Regresión logística,0.0107,0.0093,0.0121,0.0000,0.0000,[0.0088; 0.0134],0.2930,pequeño


In [8]:
icm = []  # IC BCa de las métricas principales de los 7 modelos
for k in siete:  # Cada modelo
    s = finales[k]["p_te"].astype(float)  # Puntaje
    for nombre, met in [("AUC-ROC", roc_auc_score), ("AUC-PR", average_precision_score)]:  # Métricas
        v, lo, hi = st.bca_municipios(y_te, s, mun, met, B=1000)  # BCa
        icm.append({"modelo": k[1], "métrica": nombre, "valor": v, "IC 95 % inferior": lo, "IC 95 % superior": hi})
pd.DataFrame(icm).pivot(index="modelo", columns="métrica", values=["valor", "IC 95 % inferior", "IC 95 % superior"]).round(4)

valor         IC 95 % inferior         IC 95 % superior        
métrica             AUC-PR AUC-ROC           AUC-PR AUC-ROC           AUC-PR AUC-ROC
modelo                                                                              
KNN                 0.8008  0.8853           0.7755  0.8750           0.8253  0.8953
Naive Bayes         0.6757  0.8621           0.6474  0.8510           0.7016  0.8729
Random Forest       0.8184  0.8963           0.7934  0.8867           0.8419  0.9060
Regresión logística 0.8076  0.8856           0.7824  0.8754           0.8321  0.8956
SVM                 0.8204  0.8967           0.7950  0.8868           0.8432  0.9066
XGBoost             0.8225  0.8977           0.7973  0.8882           0.8451  0.9072
Árbol de decisión   0.7920  0.8845           0.7645  0.8744           0.8167  0.8944

**Interpretación.** Entre los finalistas, DeLong solo separa con claridad a los modelos de árboles y la SVM de la regresión logística (ΔAUC ≈ 0,011–0,012, $p < 0{,}001$ tras Holm, IC BCa que excluyen el 0, delta de Cliff pequeño a mediano). **XGBoost frente a la SVM no es significativo tras Holm** ($p$ = 0,095) y **la SVM frente a Random Forest tampoco** ($p$ = 0,43). XGBoost frente a Random Forest sí es significativo ($p < 0{,}001$), pero la diferencia es de 0,0014 de AUC y el delta de Cliff es **despreciable** (0,045): con 37 600 filas, DeLong detecta diferencias sin relevancia práctica. Los intervalos BCa de AUC-PR de los tres mejores (≈ ±0,025) se superponen casi por completo. Conclusión: XGBoost es el mejor modelo puntual, pero **XGBoost, SVM y Random Forest son, en la práctica, equivalentes**; los tres son claramente superiores a los modelos lineales y de vecindad.

## 5. Regresión: secuencia jerárquica

Las pérdidas se agregan por semana (error cuadrático medio de los 376 municipios en cada una de las 100 semanas de la prueba), lo que da series temporales de pérdidas comparables entre modelos.

1. **Model Confidence Set** sobre las 28 combinaciones (α = 0,10, bootstrap estacionario por bloques).
2. **Giacomini-White** (habilidad condicional) entre los finalistas y **Clark-West** para el par anidado (persistencia ⊂ Ridge).
3. **Diebold-Mariano con corrección HLN** entre el mejor y el segundo, con **bootstrap estacionario** para el IC.
4. **SPA de Hansen**: ¿algún modelo supera genuinamente a la persistencia?
5. **d de Cohen** como tamaño del efecto.

In [9]:
y_r = te["y_reg"].to_numpy()  # Objetivo de regresión
regs = {(k[1], k[3]): finales[k]["p_te"].astype(float) for k in finales if k[0] == "reg"}  # 28 predicciones
perdidas = pd.DataFrame({f"{m} · {o}": pd.Series((y_r - f) ** 2).groupby(semanas).mean().to_numpy() for (m, o), f in regs.items()})  # 100 semanas × 28
mcs_t = st.mcs(perdidas, alfa=0.10)  # Model Confidence Set
mcs_t

,pérdida media,rango medio,p MCS,¿en el conjunto?
XGBoost · Genética (DEAP),0.3288,6.2600,1.0000,True
XGBoost · Random Search,0.3329,9.6000,0.0010,False
SVR · Genética (DEAP),0.3330,6.0200,0.1800,True
SVR · Grid Search,0.3333,6.1500,0.1800,True
Random Forest · Grid Search,0.3338,9.7800,0.0000,False
Random Forest · Bayesiana (Optuna TPE),0.3339,9.7200,0.0000,False
XGBoost · Bayesiana (Optuna TPE),0.3341,10.0900,0.0010,False
SVR · Bayesiana (Optuna TPE),0.3343,8.0500,0.0010,False
Random Forest · Genética (DEAP),0.3379,11.9800,0.0010,False
Random Forest · Random Search,0.3388,12.4400,0.0010,False


In [10]:
finalistas = mcs_t[mcs_t["¿en el conjunto?"]].index.tolist()  # Modelos en el MCS
mejor, segundo = mcs_t.index[0], mcs_t.index[1]  # Los dos de menor pérdida
filas = []  # Comparaciones dirigidas
for a in finalistas[:4]:  # Finalistas (hasta 4)
    for b in finalistas[:4]:  # Pares
        if a < b:  # Cada par una vez
            gw, p_gw = st.giacomini_white(perdidas[a], perdidas[b])  # Giacomini-White
            filas.append({"modelo A": a, "modelo B": b, "GW": gw, "p GW": p_gw})  # Fila
gwt = pd.DataFrame(filas)  # Tabla
if len(gwt):  # Holm
    o_ = np.argsort(gwt["p GW"].to_numpy())  # Orden
    gwt.loc[gwt.index[o_], "p Holm"] = np.minimum(1, np.maximum.accumulate((len(gwt) - np.arange(len(gwt))) * gwt["p GW"].to_numpy()[o_]))  # Ajuste
gwt

,modelo A,modelo B,GW,p GW,p Holm
0,SVR · Genética (DEAP),XGBoost · Genética (DEAP),10.9697,0.0041,0.0074
1,SVR · Genética (DEAP),SVR · Grid Search,25.8111,0.0000,0.0000
2,SVR · Grid Search,XGBoost · Genética (DEAP),11.2103,0.0037,0.0074


In [11]:
ridge = next(f for (m, o), f in regs.items() if m == "Ridge" and f"{m} · {o}" == mcs_t.loc[mcs_t.index.str.startswith("Ridge")].index[0])  # Mejor Ridge
cw, p_cw = st.clark_west(y_r, te["lc_l0"].to_numpy(), ridge)  # Persistencia (anidada: un solo regresor con coeficiente 1) frente a Ridge
d_med, hln, p_dm = st.diebold_mariano_perdidas(perdidas[segundo], perdidas[mejor], h=1)  # DM-HLN (positivo: el mejor tiene menos pérdida)
m_bs, lo_bs, hi_bs = st.ic_bootstrap_estacionario(perdidas[segundo] - perdidas[mejor])  # IC con bootstrap estacionario
pers = pd.Series((y_r - te["lc_l0"].to_numpy()) ** 2).groupby(semanas).mean().to_numpy()  # Pérdida semanal de la persistencia
p_spa = st.spa(pers, perdidas.to_numpy())  # SPA frente a la persistencia
pd.DataFrame({
    "prueba": ["Clark-West (persistencia ⊂ Ridge)", f"Diebold-Mariano HLN ({segundo} vs {mejor})", "SPA de Hansen (benchmark: persistencia)"],
    "estadístico": [cw, hln, np.nan], "valor p": [p_cw, p_dm, float(p_spa["consistent"])],
    "detalle": [f"H1: Ridge predice mejor que la persistencia", f"diferencia media de pérdida {d_med:.4f}, IC 95 % bootstrap estacionario [{lo_bs:.4f}; {hi_bs:.4f}]",
                f"p inferior {float(p_spa['lower']):.3f}, consistente {float(p_spa['consistent']):.3f}, superior {float(p_spa['upper']):.3f}"],
    "d de Cohen": [np.nan, st.d_cohen(perdidas[segundo], perdidas[mejor]), np.nan]})

,prueba,estadístico,valor p,detalle,d de Cohen
0,Clark-West (persistencia ⊂ Ridge),77.5897,0.0000,H1: Ridge predice mejor que la persistencia,NaN
1,Diebold-Mariano HLN (XGBoost · Random Search v...,6.3587,0.0000,"diferencia media de pérdida 0.0041, IC 95 % bo...",0.0659
2,SPA de Hansen (benchmark: persistencia),NaN,0.0000,"p inferior 0.000, consistente 0.000, superior ...",NaN


**Interpretación.**

1. **MCS** (α = 0,10): el conjunto de confianza retiene solo **3 de 28** combinaciones: XGBoost con el genético y la SVR con el genético y con la malla. Las demás son significativamente peores que la mejor.
2. **Giacomini-White:** entre los finalistas, la habilidad predictiva **condicional** sí difiere (p de Holm < 0,01): aunque en promedio son indistinguibles, uno es mejor que otro según el estado reciente (por ejemplo, semanas de subida epidémica frente a semanas tranquilas). **Clark-West** confirma que Ridge supera a la persistencia, el modelo anidado más simple ($p < 0{,}001$).
3. **Diebold-Mariano con HLN** entre los dos de menor pérdida media: la diferencia es significativa ($p < 0{,}001$, IC por bootstrap estacionario que excluye el 0), pero su **tamaño de efecto es mínimo** (d de Cohen 0,07).
4. **SPA** rechaza que ningún modelo supere a la persistencia ($p < 0{,}001$): la mejora sobre la línea base trivial es genuina y no un artefacto de haber probado 28 modelos (*data snooping*).

Igual que en clasificación: los modelos de árboles y la SVR con núcleo aproximado son los mejores y superan con claridad a la línea base, pero las diferencias entre los mejores son pequeñas.